# LeetCode 4020 — Elevator Requests I bằng Python

Link đề: https://leetcode.com/problems/elevator-requests-i/ · Độ khó: **Easy** · Chủ đề: Array, Simulation, Math

Skeleton LeetCode:

```python
class Solution:
    def elevatorRequests(self, n: int, requests: List[int]) -> int:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: thang máy bắt đầu ở tầng `0`, phục vụ các yêu cầu `requests` **theo đúng thứ tự**, mỗi tầng di chuyển mất **1 giây**; trả về tổng thời gian.
2. Nhận ra tổng thời gian là **tổng khoảng cách tuyệt đối giữa các tầng liên tiếp**, tính cả đoạn đi từ tầng `0` tới yêu cầu đầu tiên.
3. Biết dùng `abs`, `zip(prev, cur)`, `itertools.accumulate`/`pairwise` và mô phỏng từng giây.
4. Viết 4 cách (mô phỏng từng giây, vòng lặp `prev`, `zip`, `sum`/`map`) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh lỗi: quên điểm xuất phát `0`, quên `abs`, dùng `max - min` hoặc tổng các tầng, nhầm cách tính khi hai yêu cầu liên tiếp trùng tầng.

## 2. Đề bài & ví dụ

Một tòa nhà có `n` tầng, đánh số từ `0` đến `n - 1`. Mảng `requests` liệt kê các **yêu cầu tới tầng** (mỗi phần tử là một tầng). Thang máy **bắt đầu ở tầng `0`** và phục vụ các yêu cầu **theo đúng thứ tự** trong mảng. Nó di chuyển **1 tầng mỗi giây**; nếu đã đứng đúng tầng được yêu cầu thì **không tốn thời gian**; và nó bắt đầu đi tới yêu cầu kế tiếp ngay khi hoàn thành yêu cầu hiện tại. Trả về **tổng số giây** để phục vụ hết các yêu cầu.

| Ví dụ | `n`, `requests` | Output | Giải thích |
|---|---|---|---|
| 1 | `5`, `[2,1,4,3]` | `7` | `0->2`: 2s; `2->1`: 1s; `1->4`: 3s; `4->3`: 1s. Tổng `2+1+3+1 = 7` |
| 2 | `3`, `[2,0,0]` | `4` | `0->2`: 2s; `2->0`: 2s; `0->0`: 0s. Tổng `4` |

Constraints: `1 <= n <= 100`; `1 <= len(requests) <= 100`; `0 <= requests[i] <= n - 1`.

Tham số `n` chỉ cho biết phạm vi tầng; nó **không** ảnh hưởng đáp án.

In [1]:
n, requests = 5, [2, 1, 4, 3]
floor, total = 0, 0
for r in requests:
    d = abs(r - floor)
    print(f"tầng {floor} -> {r}: {d}s")
    total += d
    floor = r
print("tổng:", total)

tầng 0 -> 2: 2s
tầng 2 -> 1: 1s
tầng 1 -> 4: 3s
tầng 4 -> 3: 1s
tổng: 7


## 3. Kiến thức nền

### 3.1. `abs` và khoảng cách giữa hai tầng

Số giây đi từ tầng `a` đến tầng `b` là `abs(a - b)`: đi lên (`b > a`) hay xuống (`b < a`) đều tốn `|b - a|` giây. Đi từ tầng `a` về chính nó là `abs(0) = 0`.

In [2]:
print(abs(4 - 1), abs(1 - 4), abs(3 - 3))

3 3 0


### 3.2. Duyệt với biến "vị trí hiện tại"

Mẫu hay dùng: giữ biến `prev` (vị trí trước đó); mỗi phần tử mới `cur` cộng `abs(cur - prev)` rồi gán `prev = cur`. Khởi tạo `prev` bằng vị trí xuất phát (ở đây `0`).

In [3]:
prev, total = 0, 0
for cur in [2, 0, 0]:
    total += abs(cur - prev)
    prev = cur
print(total)

4


### 3.3. `zip` để ghép phần tử với phần tử liền trước

`zip([0] + requests, requests)` ghép `(0, r0)`, `(r0, r1)`, `(r1, r2)`, ...: mỗi cặp là (tầng trước, tầng sau). Danh sách `[0] + requests` thêm điểm xuất phát vào đầu.

In [4]:
requests = [2, 1, 4, 3]
print(list(zip([0] + requests, requests)))

[(0, 2), (2, 1), (1, 4), (4, 3)]


### 3.4. `sum` với generator và `map`

`sum(abs(a - b) for a, b in zip(...))` cộng mà không tạo list. Hoặc `sum(map(lambda p: abs(p[0] - p[1]), zip(...)))`. Hai cách tương đương; generator dễ đọc hơn.

In [5]:
pairs = list(zip([0, 2, 1, 4], [2, 1, 4, 3]))
print(sum(abs(a - b) for a, b in pairs), sum(map(lambda p: abs(p[0] - p[1]), pairs)))

7 7


### 3.5. `itertools.accumulate` (tham khảo)

`accumulate(iterable)` cho tổng chạy. Có thể dùng để lấy **thời điểm** hoàn thành sau mỗi yêu cầu: `list(accumulate(distances))`. Giá trị cuối là tổng thời gian.

In [6]:
from itertools import accumulate
dists = [2, 1, 3, 1]
print(list(accumulate(dists)))

[2, 3, 6, 7]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Thang máy phục vụ yêu cầu **lần lượt**, mỗi lần đi từ nơi vừa dừng tới tầng cần tới. Tổng thời gian là **tổng các đoạn đường** giữa các lần dừng. Đoạn đầu là từ tầng `0` tới `requests[0]`.

Tính tay `[2, 1, 4, 3]`: các điểm dừng `0, 2, 1, 4, 3` -> các đoạn `|2-0|=2`, `|1-2|=1`, `|4-1|=3`, `|3-4|=1` -> `7`.
Tính tay `[2, 0, 0]`: điểm dừng `0, 2, 0, 0` -> `2, 2, 0` -> `4`.
Tính tay `[0, 0]`: điểm dừng `0, 0, 0` -> `0, 0` -> `0`.

| điểm dừng | đoạn đường | cộng dồn |
|---|---|---|
| `0 -> 2` | 2 | 2 |
| `2 -> 1` | 1 | 3 |
| `1 -> 4` | 3 | 6 |
| `4 -> 3` | 1 | 7 |

In [7]:
def stops(requests):
    return [0] + requests

for req in ([2, 1, 4, 3], [2, 0, 0], [0, 0]):
    s = stops(req)
    segs = [abs(b - a) for a, b in zip(s, s[1:])]
    print(req, "dừng:", s, "đoạn:", segs, "tổng:", sum(segs))

[2, 1, 4, 3] dừng: [0, 2, 1, 4, 3] đoạn: [2, 1, 3, 1] tổng: 7
[2, 0, 0] dừng: [0, 2, 0, 0] đoạn: [2, 2, 0] tổng: 4
[0, 0] dừng: [0, 0, 0] đoạn: [0, 0] tổng: 0


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng giây

Cho thang máy chạy theo **từng giây**: mỗi giây thang di chuyển một tầng về phía tầng đích hiện tại; tăng đếm thời gian. Khi tới nơi thì chuyển sang yêu cầu kế (không tốn thêm giây nếu đã đúng tầng).

Đúng nhưng tốn công: thời gian lặp bằng đúng đáp án (tối đa `100 * 99`). "Nút thắt": ta đang **đếm** từng giây thay vì **tính** khoảng cách bằng một phép trừ.

In [8]:
class SolutionSimulate:
    def elevatorRequests(self, n, requests):
        floor, seconds = 0, 0
        for target in requests:
            while floor != target:
                floor += 1 if target > floor else -1        # đi một tầng mỗi giây
                seconds += 1
        return seconds

print(SolutionSimulate().elevatorRequests(5, [2, 1, 4, 3]), SolutionSimulate().elevatorRequests(3, [2, 0, 0]))

7 4


**Độ phức tạp:** Time `O(T)` với `T` là đáp án (tối đa `100 * 99`), Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** không cần mô phỏng từng giây: đoạn từ `a` đến `b` tốn đúng `abs(a - b)` giây. Tổng thời gian là **tổng các `abs` của hiệu các điểm dừng liên tiếp**:

`T = |r_0 - 0| + |r_1 - r_0| + ... + |r_{m-1} - r_{m-2}|`.

**Quy tắc:** giữ `prev` (bắt đầu `0`), với mỗi yêu cầu `cur`: `T += abs(cur - prev); prev = cur`.

**Invariant:** sau khi phục vụ `k` yêu cầu đầu, `prev = requests[k-1]` (hoặc `0` khi `k = 0`) và `T` là tổng thời gian đã dùng.

**Chú ý:** thứ tự yêu cầu **cố định**: không được sắp xếp lại để "tối ưu" (đề không cho phép). Vì thế đáp án không phải `max - min` hay tổng các tầng.

## 7. Các cách tiếp theo

### Cách 2 — Vòng lặp với biến `prev`

Duyệt từng yêu cầu, cộng `abs`.

In [9]:
class SolutionLoop:
    def elevatorRequests(self, n, requests):
        prev, total = 0, 0
        for cur in requests:
            total += abs(cur - prev)
            prev = cur
        return total

print(SolutionLoop().elevatorRequests(5, [2, 1, 4, 3]), SolutionLoop().elevatorRequests(3, [2, 0, 0]))

7 4


**Độ phức tạp:** Time `O(m)` với `m = len(requests)`, Space `O(1)`.

### Cách 3 — `zip` với danh sách có điểm xuất phát

`sum(abs(a - b) for a, b in zip([0] + requests, requests))`: một dòng. Chú ý `[0] + requests` tạo list mới `O(m)`.

In [10]:
class SolutionZip:
    def elevatorRequests(self, n, requests):
        return sum(abs(a - b) for a, b in zip([0] + requests, requests))

print(SolutionZip().elevatorRequests(5, [2, 1, 4, 3]), SolutionZip().elevatorRequests(3, [2, 0, 0]))

7 4


**Độ phức tạp:** Time `O(m)`, Space `O(m)` (list phụ).

### Cách 4 — `itertools.chain` + `pairwise`-kiểu (không tạo list phụ)

`chain([0], requests)` nối điểm xuất phát vào đầu **mà không tạo list**; ghép với `requests` bằng `zip`: `zip(chain([0], requests), requests)`. Cùng kết quả, bộ nhớ phụ `O(1)`.

In [11]:
from itertools import chain

class SolutionChain:
    def elevatorRequests(self, n, requests):
        return sum(abs(a - b) for a, b in zip(chain([0], requests), requests))

print(SolutionChain().elevatorRequests(5, [2, 1, 4, 3]), SolutionChain().elevatorRequests(3, [2, 0, 0]))

7 4


**Độ phức tạp:** Time `O(m)`, Space `O(1)` bộ nhớ phụ (trừ iterator).

## 8. TRACE từng bước

Trace Cách 2 với `requests = [2, 0, 0]`:

| yêu cầu | `prev` trước | `abs(cur - prev)` | `total` | `prev` sau |
|---|---|---|---|---|
| (đầu) | 0 | | 0 | 0 |
| 2 | 0 | 2 | 2 | 2 |
| 0 | 2 | 2 | 4 | 0 |
| 0 | 0 | 0 | 4 | 0 |

Tổng `4`.

In [12]:
def trace_loop(requests):
    prev, total = 0, 0
    print("bắt đầu: prev=0, total=0")
    for k, cur in enumerate(requests, 1):
        d = abs(cur - prev)
        total += d
        print(f"  yêu cầu {k}: {prev} -> {cur}: {d}s, total={total}")
        prev = cur
    return total

assert trace_loop([2, 1, 4, 3]) == 7
assert trace_loop([2, 0, 0]) == 4
assert trace_loop([0]) == 0

bắt đầu: prev=0, total=0
  yêu cầu 1: 0 -> 2: 2s, total=2
  yêu cầu 2: 2 -> 1: 1s, total=3
  yêu cầu 3: 1 -> 4: 3s, total=6
  yêu cầu 4: 4 -> 3: 1s, total=7
bắt đầu: prev=0, total=0
  yêu cầu 1: 0 -> 2: 2s, total=2
  yêu cầu 2: 2 -> 0: 2s, total=4
  yêu cầu 3: 0 -> 0: 0s, total=4
bắt đầu: prev=0, total=0
  yêu cầu 1: 0 -> 0: 0s, total=0


In [13]:
def trace_timeline(requests):
    t, floor, timeline = 0, 0, [(0, 0)]
    for r in requests:
        t += abs(r - floor)
        floor = r
        timeline.append((t, floor))
    print("(giây, tầng):", timeline)
    return t

assert trace_timeline([2, 1, 4, 3]) == 7

(giây, tầng): [(0, 0), (2, 2), (3, 1), (6, 4), (7, 3)]


## 9. Code nộp LeetCode

Dùng Cách 2 (vòng lặp `prev`).

In [14]:
from typing import List, Optional


class Solution:
    def elevatorRequests(self, n: int, requests: List[int]) -> int:
        prev, total = 0, 0
        for cur in requests:
            total += abs(cur - prev)
            prev = cur
        return total


print(Solution().elevatorRequests(5, [2, 1, 4, 3]), Solution().elevatorRequests(3, [2, 0, 0]))

7 4


## 10. Vì sao thuật toán đúng

1. **Một đoạn đường.** Thang máy di chuyển 1 tầng mỗi giây nên đi từ tầng `a` đến tầng `b` mất đúng `|a - b|` giây (đi thẳng, không có lý do đi vòng). Nếu `a = b` mất `0` giây, đúng như đề.
2. **Thứ tự cố định.** Đề bắt phục vụ theo **đúng thứ tự** mảng, nên các điểm dừng là `0, r_0, r_1, ..., r_{m-1}` xác định duy nhất.
3. **Cộng các đoạn.** Thang máy bắt đầu đoạn kế **ngay khi** hoàn thành đoạn hiện tại (không chờ), nên tổng thời gian bằng tổng các độ dài đoạn.
4. **Quy nạp (invariant).** Trước khi xử lý yêu cầu `k`, `prev` là tầng thang đang đứng, `total` là thời gian đã dùng. Xử lý `r_k` cộng `|r_k - prev|` và cập nhật `prev = r_k`. Sau `m` bước ta được tổng đúng.
5. Cách 1 mô phỏng từng giây (mỗi giây đúng 1 tầng) cho cùng số giây; Cách 3, 4 chỉ là biểu thức tổng khác của cùng đại lượng.

## 11. Phân tích Time / Space Complexity

Gọi `m = len(requests) <= 100`, `n <= 100`.

- Cách 1 (mô phỏng): Time `O(T)` với `T <= m * (n - 1)`, Space `O(1)`.
- Cách 2 (nộp): Time `O(m)`, Space `O(1)`.
- Cách 3 (`zip` + list): Time `O(m)`, Space `O(m)`.
- Cách 4 (`chain`): Time `O(m)`, Space `O(1)`.

Mỗi yêu cầu phải được đọc ít nhất một lần nên `O(m)` là tối ưu.

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

In [15]:
import random

solutions = [SolutionSimulate(), SolutionLoop(), SolutionZip(), SolutionChain(), Solution()]

fixed = [
    (5, [2, 1, 4, 3], 7), (3, [2, 0, 0], 4), (1, [0], 0), (2, [1], 1), (2, [1, 0, 1, 0], 4),
    (100, [99], 99), (100, [99, 0], 198), (10, [5, 5, 5], 5), (10, [0, 0, 0], 0),
    (6, [1, 2, 3, 4, 5], 5), (6, [5, 4, 3, 2, 1, 0], 10), (4, [3, 0, 3, 0], 12),
]
for n, requests, expected in fixed:
    for sol in solutions:
        assert sol.elevatorRequests(n, list(requests)) == expected, (type(sol).__name__, n, requests)
print("test cố định OK:", len(fixed), "ca")

random.seed(4020)
for _ in range(600):
    n = random.randint(1, 100)
    m = random.randint(1, 100)
    requests = [random.randint(0, n - 1) for _ in range(m)]
    outs = {sol.elevatorRequests(n, list(requests)) for sol in solutions}
    assert len(outs) == 1, (n, requests)
print("test ngẫu nhiên OK")

test cố định OK: 12 ca
test ngẫu nhiên OK


### Test thuộc tính (property test)

(1) Kết quả `>= max(requests)` (phải ít nhất đi tới tầng cao nhất). (2) Kết quả `<= m * (n - 1)`. (3) **Bất đẳng thức tam giác:** thêm một yêu cầu vào giữa không làm giảm tổng thời gian. (4) Đáp án không phụ thuộc vào `n`. (5) Không sửa `requests`.

In [16]:
random.seed(40200)
for _ in range(500):
    n = random.randint(1, 50)
    requests = [random.randint(0, n - 1) for _ in range(random.randint(1, 30))]
    snap = list(requests)
    r = Solution().elevatorRequests(n, requests)
    assert requests == snap
    assert max(requests) <= r <= len(requests) * (n - 1)
    assert Solution().elevatorRequests(n + 37, requests) == r
    k = random.randint(0, len(requests))
    extra = requests[:k] + [random.randint(0, n - 1)] + requests[k:]
    assert Solution().elevatorRequests(n, extra) >= r
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên điểm xuất phát `0`

Chỉ cộng các đoạn giữa các yêu cầu bỏ mất đoạn `0 -> requests[0]`.

In [17]:
def wrong_no_start(n, requests):
    return sum(abs(a - b) for a, b in zip(requests, requests[1:]))   # SAI: thiếu đoạn 0 -> requests[0]

print(wrong_no_start(5, [2, 1, 4, 3]), "!=", Solution().elevatorRequests(5, [2, 1, 4, 3]))
assert wrong_no_start(5, [2, 1, 4, 3]) == 5 and Solution().elevatorRequests(5, [2, 1, 4, 3]) == 7

5 != 7


### Lỗi 2: quên `abs`

Hiệu có thể âm (đi xuống). Cộng hiệu có dấu cho tổng sai (thực ra chỉ bằng `requests[-1]`: các số hạng triệt tiêu từng cặp).

In [18]:
def wrong_no_abs(n, requests):
    prev, total = 0, 0
    for cur in requests:
        total += cur - prev                                       # SAI: thiếu abs
        prev = cur
    return total

print(wrong_no_abs(5, [2, 1, 4, 3]), "!=", Solution().elevatorRequests(5, [2, 1, 4, 3]))
assert wrong_no_abs(5, [2, 1, 4, 3]) == 3 and wrong_no_abs(3, [2, 0, 0]) == 0

3 != 7


### Lỗi 3: dùng `max - min` hoặc tổng các tầng

Thứ tự phục vụ cố định, nên thang phải đi lên xuống nhiều lần. `max - min` coi như đi một chiều.

In [19]:
def wrong_range(n, requests):
    return max(requests) - min(requests)                          # SAI

def wrong_sum(n, requests):
    return sum(requests)                                          # SAI

print(wrong_range(5, [2, 1, 4, 3]), wrong_sum(5, [2, 1, 4, 3]), "!=", Solution().elevatorRequests(5, [2, 1, 4, 3]))
assert wrong_range(5, [2, 1, 4, 3]) == 3 and wrong_sum(5, [2, 1, 4, 3]) == 10

3 10 != 7


### Lỗi 4: sắp xếp `requests` để "tối ưu"

Đề **không** cho đổi thứ tự. Sắp xếp rồi tính cho kết quả nhỏ hơn đáp án thật khi các yêu cầu không tăng dần.

In [20]:
def wrong_sorted(n, requests):
    r = sorted(requests)                                          # SAI: đổi thứ tự phục vụ
    prev, total = 0, 0
    for cur in r:
        total += abs(cur - prev)
        prev = cur
    return total

print(wrong_sorted(5, [2, 1, 4, 3]), "!=", Solution().elevatorRequests(5, [2, 1, 4, 3]))
assert wrong_sorted(5, [2, 1, 4, 3]) == 4 and Solution().elevatorRequests(5, [2, 1, 4, 3]) == 7

4 != 7


### Lỗi 5: cộng 1 giây cho yêu cầu trùng tầng

Đề nói đã ở đúng tầng thì **không tốn thời gian**. Cộng `max(1, abs(...))` hay `+ 1` mỗi yêu cầu cho kết quả quá lớn.

In [21]:
def wrong_min_one(n, requests):
    prev, total = 0, 0
    for cur in requests:
        total += max(1, abs(cur - prev))                          # SAI: yêu cầu cùng tầng mất 0s
        prev = cur
    return total

print(wrong_min_one(3, [2, 0, 0]), "!=", Solution().elevatorRequests(3, [2, 0, 0]))
assert wrong_min_one(3, [2, 0, 0]) == 5 and Solution().elevatorRequests(3, [2, 0, 0]) == 4

5 != 4


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đi lần lượt qua các điểm theo **thứ tự cho trước**" -> **tổng khoảng cách giữa các điểm liên tiếp** (có cả đoạn từ điểm xuất phát).
- Mẫu `prev` + `abs(cur - prev)` rất hay gặp: "Minimum Time Visiting All Points" (1266), "Time to type a word" (1165).
- Tham số `n` có thể là "đồ trang trí" (không ảnh hưởng đáp án) — hãy nhận ra để khỏi lo.
- Thêm điểm xuất phát vào đầu: `[0] + requests` hoặc khởi tạo `prev = 0`.
- Kiểm tra bằng **bất đẳng thức tam giác**: chen thêm điểm không bao giờ làm tổng giảm.
- Thử ca: một yêu cầu, yêu cầu ở tầng `0`, yêu cầu lặp lại cùng một tầng.

## 15. Biến thể của bài

1. **Minimum Time Visiting All Points** (1266): đi qua các điểm 2D theo thứ tự (khoảng cách Chebyshev).
2. **Minimum Time to Type Word Using Special Typewriter** (1974): quay vòng 26 chữ cái.
3. **Elevator Requests II** (nếu có): nhiều yêu cầu, có thể sắp xếp thứ tự phục vụ để tối ưu.
4. Biến thể: thang có **tốc độ** khác nhau (`v` giây mỗi tầng) hoặc mất `k` giây mở cửa mỗi lần dừng.
5. Biến thể: được **sắp xếp lại** thứ tự yêu cầu để tối thiểu hóa tổng thời gian (khi đó là bài kiểu "quét" / SCAN).
6. Biến thể: thang bắt đầu ở tầng `s` bất kỳ thay vì `0`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng từng giây | thang đi 1 tầng mỗi giây | `O(T)` | `O(1)` | Hiểu đề, chậm nếu `T` lớn |
| 2. Vòng lặp `prev` | cộng `abs(cur - prev)` | `O(m)` | `O(1)` | **Nộp bài** |
| 3. `zip([0] + requests, requests)` | một dòng | `O(m)` | `O(m)` | Gọn |
| 4. `zip(chain([0], requests), requests)` | không tạo list phụ | `O(m)` | `O(1)` | Dữ liệu lớn |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().elevatorRequests(10, [3, 3, 1])` và `Solution().elevatorRequests(4, [3, 0, 3])` ra gì?

**Bài 2 (tính tay):** với `n = 8`, `requests = [5, 2, 7, 7, 0]`, liệt kê các đoạn và tổng.

**Bài 3 (điền TODO):** hoàn thành `elevator_with_door(n, requests, k)`: mỗi lần **dừng ở tầng yêu cầu** (kể cả khi đã đứng đúng tầng) mất thêm `k` giây mở cửa.

**Bài 4 (tự viết lại):** viết `timeline(requests)` trả về list thời điểm hoàn thành từng yêu cầu (tổng cộng dồn).

**Bài 5 (mini challenge):** viết `elevator_sorted_best(n, requests)`: nếu được **sắp xếp lại** yêu cầu, thời gian nhỏ nhất là bao nhiêu? (gợi ý: đi một chiều tới tầng cao nhất).

In [22]:
# Bài 1: đoán rồi chạy
print(Solution().elevatorRequests(10, [3, 3, 1]), Solution().elevatorRequests(4, [3, 0, 3]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def elevator_with_door_todo(n, requests, k):
    # TODO: thời gian di chuyển + k * (số yêu cầu)
    return 0

5 9


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: điểm dừng `0,3,3,1` và `0,3,0,3`. Bài 2: điểm dừng `0,5,2,7,7,0`.

**Mức 2 (vừa):** Bài 3: `Solution().elevatorRequests(...) + k * len(requests)`. Bài 4: `itertools.accumulate` của các đoạn. Bài 5: thang chỉ cần đi từ `0` lên tầng cao nhất (không quay lại) vì bỏ qua hậu quả quay về.

**Mức 3 (gần đáp án):** Bài 1: `5` (3 + 0 + 2) và `9` (3 + 3 + 3). Bài 2: đoạn `5, 3, 5, 0, 7` -> `20`. Bài 3: `(5, [2, 1, 4, 3], 2) -> 15`. Bài 4: `[2, 1, 4, 3] -> [2, 3, 6, 7]`. Bài 5: `max(requests)`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert Solution().elevatorRequests(10, [3, 3, 1]) == 5 and Solution().elevatorRequests(4, [3, 0, 3]) == 9
assert Solution().elevatorRequests(8, [5, 2, 7, 7, 0]) == 20

# Bài 3
def elevator_with_door(n, requests, k):
    return Solution().elevatorRequests(n, requests) + k * len(requests)

assert elevator_with_door(5, [2, 1, 4, 3], 2) == 15 and elevator_with_door(1, [0], 3) == 3

# Bài 4
def timeline(requests):
    t, prev, out = 0, 0, []
    for cur in requests:
        t += abs(cur - prev)
        prev = cur
        out.append(t)
    return out

assert timeline([2, 1, 4, 3]) == [2, 3, 6, 7] and timeline([0, 0]) == [0, 0]
assert timeline([2, 0, 0])[-1] == Solution().elevatorRequests(3, [2, 0, 0])

# Bài 5
def elevator_sorted_best(n, requests):
    return max(requests)

assert elevator_sorted_best(5, [2, 1, 4, 3]) == 4
assert sum(abs(b - a) for a, b in zip([0] + sorted([2, 1, 4, 3]), sorted([2, 1, 4, 3]))) == 4
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def elevatorRequests(self, n: int, requests: List[int]) -> int:
        prev, total = 0, 0              # thang bắt đầu ở tầng 0
        for cur in requests:
            total += abs(cur - prev)    # |tầng mới - tầng cũ| giây
            prev = cur
        return total
```

Khung chung: "đi qua các điểm theo thứ tự cho trước" -> **tổng khoảng cách liên tiếp**, nhớ điểm xuất phát.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] `prev` khởi tạo `0` (điểm xuất phát), có `abs`, **không** sắp xếp lại `requests`.
- [ ] Đã thử một yêu cầu `[0]` (kết quả `0`), yêu cầu lặp cùng tầng, và đi xuống nhiều tầng.